[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/06_Validation_and_QA.ipynb)

# Notebook 06 — Validation and QA
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Resolve GPS-review records, verify fuzzy matches, flag yield/population outliers, and confirm ward boundary alignment before final outputs.

---
## QA tasks
| Task | Records | Priority |
|------|---------|----------|
| GPS-review: shared coordinates | 146 | High — blocks coverage gap map |
| Low-confidence fuzzy matches (<0.90) | ~60 | High — may misplace boreholes |
| Yield outlier review | ~15 | Medium — may inflate scoring |
| Population outlier review | ~10 | Medium — affects WASI component 4 |
| Ward name alignment (GADM vs dataset) | 40 wards | High — blocks spatial joins |

---
## Outputs
- `kitui_gps_review_flags.csv` — records to manually verify in the field
- `kitui_outlier_flags.csv` — yield and population records requiring verification
- `kitui_ward_name_crosswalk.csv` — GADM → dataset name alignment table
- Inline review tables for analyst decision-making

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install pandas geopandas openpyxl fuzzywuzzy python-Levenshtein -q

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from fuzzywuzzy import fuzz, process
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
WGS84 = 'EPSG:4326'

import os
os.makedirs(OUT, exist_ok=True)

print('Setup complete')

In [ ]:
# ── 1. Load full master dataset ───────────────────────────────────────────────

df = pd.read_excel(
    DRIVE + 'Kitui_Boreholes_Master_Dataset.xlsx',
    sheet_name='B_Spatial_Analysis',
    header=2
)

gdf = gpd.GeoDataFrame(
    df[df['GPS_Available'] == True].copy(),
    geometry=[Point(xy) for xy in zip(
        df.loc[df['GPS_Available'] == True, 'Longitude'],
        df.loc[df['GPS_Available'] == True, 'Latitude']
    )],
    crs=WGS84
)

print(f'Total records: {len(df)}')
print(f'GPS available: {len(gdf)}')
print()
print('GPS quality breakdown:')
print(df['GPS_Quality'].value_counts().to_string())

In [ ]:
# ── 2. GPS-review: Shared coordinates ─────────────────────────────────────────
#
# GPS_Shared_Flag = True means a fuzzy-matched record was assigned GPS from
# another borehole. Multiple boreholes pointing to the same coordinate
# inflate coverage and distort the distance layer.

shared = gdf[gdf['GPS_Shared_Flag'] == True].copy()
print(f'Records with shared GPS coordinates: {len(shared)}')

# Group by coordinate to see which clusters need review
shared['coord_key'] = shared['Longitude'].round(5).astype(str) + ',' + \
                      shared['Latitude'].round(5).astype(str)

coord_groups = shared.groupby('coord_key').agg(
    n_boreholes=('Borehole_ID', 'count'),
    borehole_ids=('Borehole_ID', lambda x: ', '.join(x)),
    borehole_names=('Borehole_Name', lambda x: ' | '.join(x)),
    ward=('Ward', 'first'),
    match_confidences=('Match_Confidence', lambda x: ', '.join(x.round(2).astype(str)))
).reset_index()

print(f'Unique coordinate clusters: {len(coord_groups)}')
print(f'Clusters with 2+ boreholes: {(coord_groups["n_boreholes"] >= 2).sum()}')
print()
print('Top 10 most crowded coordinates:')
print(coord_groups.nlargest(10, 'n_boreholes')
      [['ward', 'n_boreholes', 'borehole_names', 'match_confidences']]
      .to_string(index=False))

In [ ]:
# ── 3. Low-confidence fuzzy matches ──────────────────────────────────────────
#
# Fuzzy matches with Match_Confidence < 0.90 may have assigned GPS to the
# wrong borehole. Flag for manual review against physical site list.

LOW_CONF_THRESHOLD = 0.90

low_conf = df[
    (df['Match_Method'] == 'fuzzy') &
    (df['Match_Confidence'] < LOW_CONF_THRESHOLD)
].copy()

print(f'Low-confidence fuzzy matches (< {LOW_CONF_THRESHOLD}): {len(low_conf)}')
print()

# Categorise by confidence level
bins   = [0, 0.70, 0.80, 0.85, 0.90]
labels = ['<70% — high risk', '70–79% — review', '80–84% — review', '85–89% — check']
low_conf['Conf_Band'] = pd.cut(low_conf['Match_Confidence'], bins=bins, labels=labels)
print(low_conf['Conf_Band'].value_counts().sort_index().to_string())
print()
print('High-risk matches (confidence < 0.70):')
high_risk = low_conf[low_conf['Match_Confidence'] < 0.70]
if len(high_risk) > 0:
    print(high_risk[['Borehole_ID', 'Borehole_Name', 'Ward',
                      'Match_Confidence', 'Latitude', 'Longitude']]
          .sort_values('Match_Confidence')
          .to_string(index=False))
else:
    print('None — all fuzzy matches are >= 70% confidence')

In [ ]:
# ── 4. Geographic boundary validation ─────────────────────────────────────────
#
# Boreholes should fall inside Kitui County. Any points outside are likely
# data entry errors (swapped lat/lon, wrong decimal place, etc.)

KITUI_LAT_RANGE = (-2.2, -0.2)
KITUI_LON_RANGE = (37.6, 39.1)

out_of_bounds = gdf[
    ~(
        (gdf['Latitude'].between(*KITUI_LAT_RANGE)) &
        (gdf['Longitude'].between(*KITUI_LON_RANGE))
    )
]

print(f'Points outside Kitui bounding box: {len(out_of_bounds)}')
if len(out_of_bounds) > 0:
    print(out_of_bounds[['Borehole_ID', 'Borehole_Name', 'Ward',
                          'Latitude', 'Longitude', 'GPS_Quality']]
          .to_string(index=False))
    print()
    print('ACTION: These records need GPS correction before use in spatial analysis.')
else:
    print('All GPS points are within the Kitui bounding box.')

# Latitude/longitude swap check
possible_swap = gdf[
    (gdf['Latitude'].between(*KITUI_LON_RANGE)) &
    (gdf['Longitude'].between(*KITUI_LAT_RANGE))
]
if len(possible_swap) > 0:
    print(f'\nPossible lat/lon swaps: {len(possible_swap)}')
    print(possible_swap[['Borehole_ID','Borehole_Name','Ward','Latitude','Longitude']]
          .to_string(index=False))

In [ ]:
# ── 5. Yield outlier detection ─────────────────────────────────────────────────
#
# Very high yield values (>50 m³/hr) likely represent scheme-level totals
# rather than single borehole yields. Flag for verification.
#
# See data_dictionary.md: 'Yield_m3_hr — Verify if >50 (may be scheme-level total)'

yield_data = df[df['Yield_m3_hr'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(yield_data['Yield_m3_hr'].clip(upper=100), bins=40,
             color='#2E75B6', edgecolor='white')
axes[0].axvline(50, color='red', linestyle='--', label='50 m³/hr threshold')
axes[0].set_xlabel('Yield (m³/hr)')
axes[0].set_ylabel('Count')
axes[0].set_title('Yield distribution (capped at 100 m³/hr for display)')
axes[0].legend()

# Box plot by management type
yield_data[yield_data['Yield_m3_hr'] < 100].boxplot(
    column='Yield_m3_hr', by='Management_Type',
    ax=axes[1], rot=45
)
axes[1].set_title('Yield by management type')
axes[1].set_xlabel('Management type')
axes[1].set_ylabel('Yield (m³/hr)')
plt.suptitle('')
plt.tight_layout()
plt.show()

# Flag outliers
HIGH_YIELD = 50
yield_outliers = yield_data[yield_data['Yield_m3_hr'] > HIGH_YIELD]

print(f'\nYield data summary:')
print(f'  Records with yield data: {len(yield_data)}')
print(f'  Mean yield: {yield_data["Yield_m3_hr"].mean():.1f} m³/hr')
print(f'  Median yield: {yield_data["Yield_m3_hr"].median():.1f} m³/hr')
print(f'  Max yield: {yield_data["Yield_m3_hr"].max():.1f} m³/hr')
print(f'\nYield outliers (>{HIGH_YIELD} m³/hr): {len(yield_outliers)}')
if len(yield_outliers) > 0:
    print(yield_outliers[['Borehole_ID', 'Borehole_Name', 'Ward',
                           'Yield_m3_hr', 'Is_Scheme_Entry', 'Management_Type']]
          .sort_values('Yield_m3_hr', ascending=False)
          .to_string(index=False))

In [ ]:
# ── 6. Population outlier detection ───────────────────────────────────────────
#
# Population_Served_HHs > 3000 HH may represent scheme-level totals.
# See data_dictionary.md: 'Verify if >3,000'

pop_data = df[df['Population_Served_HHs'].notna() & (df['Population_Served_HHs'] > 0)].copy()

HIGH_POP = 3000
pop_outliers = pop_data[pop_data['Population_Served_HHs'] > HIGH_POP]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pop_data['Population_Served_HHs'].clip(upper=5000), bins=50,
        color='#2E75B6', edgecolor='white')
ax.axvline(HIGH_POP, color='red', linestyle='--', label=f'{HIGH_POP} HH threshold')
ax.set_xlabel('Population served (HH)')
ax.set_ylabel('Count')
ax.set_title('Population served distribution (capped at 5,000 for display)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Population data summary:')
print(f'  Records with population data: {len(pop_data)}')
print(f'  Mean: {pop_data["Population_Served_HHs"].mean():.0f} HH')
print(f'  Median: {pop_data["Population_Served_HHs"].median():.0f} HH')
print(f'  Max: {pop_data["Population_Served_HHs"].max():.0f} HH')
print(f'\nPopulation outliers (>{HIGH_POP} HH): {len(pop_outliers)}')
if len(pop_outliers) > 0:
    print(pop_outliers[['Borehole_ID', 'Borehole_Name', 'Ward',
                         'Population_Served_HHs', 'Is_Scheme_Entry', 'Management_Type']]
          .sort_values('Population_Served_HHs', ascending=False)
          .to_string(index=False))

In [ ]:
# ── 7. Ward name alignment — dataset vs GADM boundary file ────────────────────
#
# Spatial joins between the borehole dataset and ward boundaries will fail
# silently if ward names don't match exactly.
# This cell compares the 40 ward names in the master dataset against the
# GADM boundary file and produces a crosswalk table for any mismatches.

# Dataset ward names
dataset_wards = sorted(df['Ward'].dropna().unique().tolist())

# GADM ward names — load from boundary file
try:
    wards_shp = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp')
    # Try common GADM column names
    for col in ['NAME_3', 'ADM3_EN', 'Ward', 'ward_name', 'NAME']:
        if col in wards_shp.columns:
            gadm_wards = sorted(wards_shp[col].dropna().unique().tolist())
            print(f'Ward name column in shapefile: {col}')
            break
    else:
        gadm_wards = []
        print(f'Could not find ward name column. Available columns: {wards_shp.columns.tolist()}')
except Exception as e:
    gadm_wards = []
    print(f'Could not load ward shapefile: {e}')
    print('Download Kitui GADM Level 3 from: https://gadm.org/download_country.html')

print(f'\nDataset wards: {len(dataset_wards)}')
print(f'GADM wards:    {len(gadm_wards)}')

In [ ]:
# ── 8. Build ward name crosswalk ──────────────────────────────────────────────

if gadm_wards:
    crosswalk = []
    exact_matches = []
    no_matches    = []

    for dw in dataset_wards:
        if dw in gadm_wards:
            crosswalk.append({'Dataset_Ward': dw, 'GADM_Ward': dw,
                              'Match_Type': 'Exact', 'Confidence': 1.0})
            exact_matches.append(dw)
        else:
            # Fuzzy match
            best_match, score = process.extractOne(dw, gadm_wards, scorer=fuzz.token_sort_ratio)
            if score >= 70:
                crosswalk.append({'Dataset_Ward': dw, 'GADM_Ward': best_match,
                                  'Match_Type': 'Fuzzy', 'Confidence': score / 100})
            else:
                crosswalk.append({'Dataset_Ward': dw, 'GADM_Ward': None,
                                  'Match_Type': 'No match', 'Confidence': score / 100})
                no_matches.append(dw)

    crosswalk_df = pd.DataFrame(crosswalk)

    print('Ward name alignment results:')
    print(crosswalk_df['Match_Type'].value_counts().to_string())
    print()

    fuzzy_matches = crosswalk_df[crosswalk_df['Match_Type'] == 'Fuzzy']
    if len(fuzzy_matches) > 0:
        print('Fuzzy-matched wards — verify these manually:')
        print(fuzzy_matches.to_string(index=False))

    if no_matches:
        print(f'\nNo-match wards — need manual resolution:')
        for w in no_matches:
            print(f'  {w}')

    crosswalk_df.to_csv(OUT + 'kitui_ward_name_crosswalk.csv', index=False)
    print(f'\nCrosswalk saved: {OUT}kitui_ward_name_crosswalk.csv')
    print('Add this crosswalk to merge calls in Notebooks 02–05 where GADM names differ.')
else:
    print('Skipping crosswalk — GADM shapefile not loaded.')
    print('Once boundaries are available, re-run this cell.')

In [ ]:
# ── 9. Export QA flags ────────────────────────────────────────────────────────

# Combined GPS review flag table
gps_flags = pd.concat([
    shared[['Borehole_ID','Borehole_Name','Ward','Sub_County',
            'Latitude','Longitude','GPS_Quality','Match_Method','Match_Confidence']]
    .assign(Flag_Type='Shared GPS coordinates'),
    low_conf[['Borehole_ID','Borehole_Name','Ward','Sub_County',
              'Latitude','Longitude','GPS_Quality','Match_Method','Match_Confidence']]
    .assign(Flag_Type='Low confidence fuzzy match'),
], ignore_index=True)

if len(out_of_bounds) > 0:
    gps_flags = pd.concat([
        gps_flags,
        out_of_bounds[['Borehole_ID','Borehole_Name','Ward','Sub_County',
                        'Latitude','Longitude','GPS_Quality','Match_Method','Match_Confidence']]
        .assign(Flag_Type='Out of bounds coordinates')
    ], ignore_index=True)

gps_flags['Action_Required'] = gps_flags['Flag_Type'].map({
    'Shared GPS coordinates':       'Verify unique borehole location in field or on mWater',
    'Low confidence fuzzy match':   'Confirm borehole name and GPS against physical site list',
    'Out of bounds coordinates':    'Correct GPS — likely data entry error',
})

gps_flags.to_csv(OUT + 'kitui_gps_review_flags.csv', index=False)
print(f'GPS review flags saved: {OUT}kitui_gps_review_flags.csv')
print(f'Total records flagged: {len(gps_flags)}')

# Outlier table
outlier_records = []
if len(yield_outliers) > 0:
    yield_out = yield_outliers[['Borehole_ID','Borehole_Name','Ward',
                                 'Yield_m3_hr','Is_Scheme_Entry']].copy()
    yield_out['Flag_Type'] = 'High yield'
    yield_out['Flag_Value'] = yield_out['Yield_m3_hr']
    yield_out['Threshold'] = HIGH_YIELD
    outlier_records.append(yield_out)

if len(pop_outliers) > 0:
    pop_out = pop_outliers[['Borehole_ID','Borehole_Name','Ward',
                              'Population_Served_HHs','Is_Scheme_Entry']].copy()
    pop_out['Flag_Type'] = 'High population'
    pop_out['Flag_Value'] = pop_out['Population_Served_HHs']
    pop_out['Threshold'] = HIGH_POP
    outlier_records.append(pop_out)

if outlier_records:
    all_outliers = pd.concat(outlier_records, ignore_index=True)
    all_outliers.to_csv(OUT + 'kitui_outlier_flags.csv', index=False)
    print(f'Outlier flags saved: {OUT}kitui_outlier_flags.csv ({len(all_outliers)} records)')
else:
    print('No outliers above thresholds — no outlier file written')

In [ ]:
# ── 10. QA summary ────────────────────────────────────────────────────────────

total_bh = len(df)
gps_verified  = (df['GPS_Quality'] == 'Verified').sum()
needs_review  = (df['GPS_Quality'] == 'Needs Review - Shared Coordinates').sum()
low_conf_ct   = (df['GPS_Quality'] == 'Low Confidence Match').sum()
no_gps        = (df['GPS_Quality'] == 'No GPS').sum()

print('══ QA SUMMARY ═══════════════════════════════════════════════════')
print(f'Total boreholes:                    {total_bh}')
print(f'GPS Verified (safe for analysis):   {gps_verified} ({gps_verified/total_bh*100:.1f}%)')
print(f'Needs GPS review (shared coords):   {needs_review} ({needs_review/total_bh*100:.1f}%)')
print(f'Low confidence match:               {low_conf_ct} ({low_conf_ct/total_bh*100:.1f}%)')
print(f'No GPS (field collection needed):   {no_gps} ({no_gps/total_bh*100:.1f}%)')
print(f'Out of bounds coordinates:          {len(out_of_bounds)}')
print(f'Yield outliers (>{HIGH_YIELD} m³/hr): {len(yield_outliers)}')
print(f'Population outliers (>{HIGH_POP} HH): {len(pop_outliers)}')
print('═════════════════════════════════════════════════════════════════')
print()
print('OUTPUTS:')
print(f'  {OUT}kitui_gps_review_flags.csv')
print(f'  {OUT}kitui_outlier_flags.csv')
print(f'  {OUT}kitui_ward_name_crosswalk.csv')
print()
print('── Notebook 06 complete ──────────────────────────────────────────')
print('Next: Run Notebook 07 (Report Figures) after all analysis is complete')